<!--
    Copyright (C) 2026 The AiQuokkaWannabe Authors <https://github.com/ai-quokka-wannabe/iso-lab>

    This program is free software: you can redistribute it and/or modify it under the terms of
    the GNU General Public License as published by the Free Software Foundation, either version
    3 of the License, or (at your option) any later version.

    This program is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY;
    without even the implied warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.
    See the GNU General Public License for more details.

    You should have received a copy of the GNU General Public License along with this program.
    If not, see <https://www.gnu.org/licenses/>.
-->

# The Locomotion States in Rust Against the Published Ones

The check written down in the journal of 2026-10-15, *The Locomotion States in Rust*, before it ran: each record's
published `midbody_speed` and `length` are fed to the Rust binary `motion-modes`, and its states are compared with the
published `motion_modes` (1 forward, -1 backward, 0 paused, undefined a class of its own) on the seventeen records of
the speed check. The Rust counts as reproducing the rule if the two agree on at least 99% of frames in every record.

Run from the repository's root with the block-1 files in `data/raw/owmd-n2/` (fetched by `fetch-block`, which names
a file only once its MD5 matches) and the Rust toolchain installed; the notebook builds and calls the binary.


In [1]:
import os, subprocess
import numpy as np
import h5py

os.chdir(os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd())
DIRECTORY, SUFFIX = "data/raw/owmd-n2", "_features.hdf5"
# The seventeen records of the speed check of 2026-10-14, pinned so that a re-run checks the same set.
records = ["1006562", "1007123", "1007405", "1008698", "1011246", "1011901", "1013148", "1014831", "1020507",
           "1021835", "1024910", "1025254", "1025397", "1029332", "1030984", "1031252", "1032937"]
missing = [record for record in records if not os.path.exists(f"{DIRECTORY}/{record}{SUFFIX}")]
assert not missing, f"fetch these first: {missing}"
subprocess.run(["cargo", "build", "--quiet", "--release", "--bin", "motion-modes"], check=True)
print(len(records), "records checked")
print("h5py", h5py.__version__, "numpy", np.__version__)

17 records checked
h5py 3.16.0 numpy 2.5.4


In [2]:
def text(value):
    # repr round-trips a double exactly, and Rust parses it back to the same double.
    return "NaN" if np.isnan(value) else repr(float(value))

rows = []
for record in records:
    with h5py.File(f"{DIRECTORY}/{record}{SUFFIX}", "r") as file:
        table = file["/features_timeseries"]
        fps = float(table.attrs["fps"])
        speed, length = table["midbody_speed"].astype(float), table["length"].astype(float)
        published = table["motion_modes"].astype(float)
    lines = "".join(f"{text(s)}\t{text(l)}\n" for s, l in zip(speed, length))
    out = subprocess.run(["target/release/motion-modes", repr(fps)], input=lines, check=True, capture_output=True,
                         text=True).stdout
    ours = np.array([float(line) for line in out.splitlines()])
    assert len(ours) == len(published), (record, len(ours), len(published))
    code = lambda modes: np.where(np.isnan(modes), 9, modes).astype(int)
    a, b = code(ours), code(published)
    pairs = {}
    for o, t in zip(a[a != b], b[a != b]):
        pairs[(int(o), int(t))] = pairs.get((int(o), int(t)), 0) + 1
    rows.append((record, len(a), np.mean(a == b), dict(sorted(pairs.items()))))
print("record   frames   agreement  disagreements (ours, published; 9 undefined)")
for record, n, agreement, pairs in rows:
    print(f"{record}  {n:7d}  {agreement:.5f}    {pairs}")
print("frames in all:", sum(row[1] for row in rows))
print("every record at least 99%:", all(row[2] >= 0.99 for row in rows))

record   frames   agreement  disagreements (ours, published; 9 undefined)
1006562    26998  1.00000    {}
1007123    26997  1.00000    {}
1007405    26998  1.00000    {}
1008698    24055  1.00000    {}
1011246   215850  1.00000    {}
1011901    26991  1.00000    {}
1013148    26999  1.00000    {}
1014831    26998  1.00000    {}
1020507    26998  1.00000    {}
1021835    26998  1.00000    {}
1024910    23017  1.00000    {}
1025254    23077  1.00000    {}
1025397    23431  1.00000    {}
1029332    23114  1.00000    {}
1030984    22711  1.00000    {}
1031252    22877  1.00000    {}
1032937     8205  1.00000    {}
frames in all: 602314
every record at least 99%: True
